In [13]:
import pandas as pd
import json
import os
import glob
from google.cloud import bigquery
from datetime import datetime
import pytz
from io import BytesIO

In [14]:
# --- 1. CONFIGURAÇÕES DE ACESSO ---
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "theme-park-queue-data-f2e1d4785d38.json"
TABLE_ID = "theme-park-queue-data.theme_park_queues.historical-data"

In [15]:
# IDs e Lands do Beto Carrero
BCW_LANDS = {
    'Vila Germânica': [11329, 11366, 11332],
    'Ilha dos Piratas': [11340],
    'TriplikLand': [11330, 11328, 11373, 11326, 11331],
    'Cowboyland': [13872, 11368, 11367, 11444],
    'Mundo Animal': [11358],
    'Nerf Mania': [12325, 12326],
    'Aventura Radical': [11327, 11335, 11336],
    'Madagascar': [11338],
    'Terra da Fantasia': [11344],
    'Hot Wheels': [11459, 11334, 15407]
}

ID_TO_LAND = {rid: land for land, ids in BCW_LANDS.items() for rid in ids}
ALLOWED_IDS = list(ID_TO_LAND.keys())

# Mapeamento de Correção de Nomes
NAME_FIXES = {
    "TURBO DRIVE": "Turbo Drive",
    "SUPER SOAKER SPLASH": "Super Soaker Splash",
    "SPIN BLAST": "Spin Blast"
}

In [16]:
def process_file(file_path):
    try:
        df = pd.read_csv(file_path)
        
        # 1. Filtrar IDs mecânicos
        df = df[df['id'].isin(ALLOWED_IDS)].copy()
        if df.empty: return None
        
        # 2. Tratamento de Horário (10h às 19h Brasília)
        # Assume que collected_at já está em UTC
        df['collected_at'] = pd.to_datetime(df['collected_at'])
        tz_br = pytz.timezone("America/Sao_Paulo")
        
        def is_open_hours(dt_utc):
            dt_local = dt_utc.replace(tzinfo=pytz.UTC).astimezone(tz_br)
            return 10 <= dt_local.hour < 19

        df = df[df['collected_at'].apply(is_open_hours)]
        if df.empty: return None

        # 3. Regras de Negócio: wait_time 0 -> Closed | Outliers > 300
        df.loc[df['wait_time'] == 0, 'is_open'] = False
        df = df[df['wait_time'] <= 300]
        
        # 4. Lands e Nomes
        df['land'] = df['id'].map(ID_TO_LAND)
        
        def fix_name(n):
            n_up = str(n).upper()
            for key, val in NAME_FIXES.items():
                if key in n_up: return val
            return n
        
        df['name'] = df['name'].apply(fix_name)
        
        # 5. Formatar para BigQuery
        bq_df = pd.DataFrame({
            "park_id": 319,
            "park_name": "Beto Carrero World",
            "land": df['land'],
            "ride_id": df['id'],
            "ride_name": df['name'],
            "wait_time": df['wait_time'],
            "is_open": df['is_open'],
            "timestamp_utc": df['collected_at'].dt.strftime('%Y-%m-%d %H:%M:%S'),
            "timezone": "America/Sao_Paulo"
        })
        return bq_df
    except Exception as e:
        print(f"Erro no arquivo {file_path}: {e}")
        return None

In [17]:
def upload_to_bq(full_df):
    client = bigquery.Client()
    print(f"🚀 Iniciando upload de {len(full_df)} linhas...")
    
    jsonl_data = full_df.to_json(orient='records', lines=True)
    file_obj = BytesIO(jsonl_data.encode('utf-8'))
    
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition="WRITE_APPEND",
    )
    
    load_job = client.load_table_from_file(file_obj, TABLE_ID, job_config=job_config)
    load_job.result()
    print(f"✅ Sucesso! Dados inseridos em {TABLE_ID}")

In [20]:
if __name__ == "__main__":
    # Busca recursiva em data-recovered (pega todas as subpastas 2026-04, 05...)
    path_pattern = os.path.join("data-recovered", "**", "beto_carrero_world_2026-*.csv")
    files = glob.glob(path_pattern, recursive=True)
    
    all_data = []
    start_recovery = datetime(2026, 4, 9)
    end_recovery = datetime(2026, 5, 11)

    for f in files:
        # Extrai a data do nome do arquivo para o filtro
        filename = os.path.basename(f)
        date_str = filename.replace("beto_carrero_world_", "").replace(".csv", "")
        file_date = datetime.strptime(date_str, "%Y-%m-%d")

        if start_recovery <= file_date <= end_recovery:
            processed = process_file(f)
            if processed is not None:
                all_data.append(processed)
                print(f"✔ Carregado: {filename}")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        upload_to_bq(final_df)
    else:
        print("Nenhum dado encontrado no intervalo ou pasta especificada.")

✔ Carregado: beto_carrero_world_2026-04-09.csv
✔ Carregado: beto_carrero_world_2026-04-10.csv
✔ Carregado: beto_carrero_world_2026-04-11.csv
✔ Carregado: beto_carrero_world_2026-04-12.csv
✔ Carregado: beto_carrero_world_2026-04-13.csv
✔ Carregado: beto_carrero_world_2026-04-14.csv
✔ Carregado: beto_carrero_world_2026-04-15.csv
✔ Carregado: beto_carrero_world_2026-04-16.csv
✔ Carregado: beto_carrero_world_2026-04-17.csv
✔ Carregado: beto_carrero_world_2026-04-18.csv
✔ Carregado: beto_carrero_world_2026-04-19.csv
✔ Carregado: beto_carrero_world_2026-04-20.csv
✔ Carregado: beto_carrero_world_2026-04-21.csv
✔ Carregado: beto_carrero_world_2026-04-22.csv
✔ Carregado: beto_carrero_world_2026-04-23.csv
✔ Carregado: beto_carrero_world_2026-04-24.csv
✔ Carregado: beto_carrero_world_2026-04-25.csv
✔ Carregado: beto_carrero_world_2026-04-26.csv
✔ Carregado: beto_carrero_world_2026-04-27.csv
✔ Carregado: beto_carrero_world_2026-04-28.csv
✔ Carregado: beto_carrero_world_2026-04-29.csv
✔ Carregado: 